[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_10_L2_Dropout_and_Early_Stopping.ipynb)


# Experiment 10: L2, Dropout and Early Stopping

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 10 – L2, Dropout and Early Stopping

In [ ]:
# Pima-like synthetic diabetes data, ~500 records
import tensorflow as tf
rng=np.random.default_rng(10); X=rng.normal(size=(520,8))
risk=1.2*X[:,1]+.8*X[:,5]+.4*X[:,0]+rng.normal(size=520)
y=(risk>np.quantile(risk,.55)).astype(int)
X=StandardScaler().fit_transform(X); Xa,Xtest,ya,ytest=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
def make_model(kind="base",strength=.001,drop=.3):
    reg=tf.keras.regularizers.l2(strength) if kind=="l2" else None
    layers=[tf.keras.layers.Input((8,)),tf.keras.layers.Dense(128,activation="relu",kernel_regularizer=reg)]
    if kind=="drop": layers.append(tf.keras.layers.Dropout(drop))
    layers += [tf.keras.layers.Dense(64,activation="relu",kernel_regularizer=reg)]
    if kind=="drop": layers.append(tf.keras.layers.Dropout(drop))
    layers.append(tf.keras.layers.Dense(1,activation="sigmoid"))
    m=tf.keras.Sequential(layers); m.compile("adam","binary_crossentropy",metrics=["accuracy"]); return m
results=[]
for kind,param in [("base",0),("l2",.0001),("l2",.001),("l2",.01),("drop",.2),("drop",.3),("drop",.5),("early",0)]:
    m=make_model("base" if kind=="early" else kind, strength=param if kind=="l2" else .001,drop=param if kind=="drop" else .3)
    cb=[tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)] if kind=="early" else []
    hist=m.fit(Xa,ya,epochs=60,batch_size=32,validation_split=.2,verbose=0,callbacks=cb)
    tr=m.evaluate(Xa,ya,verbose=0)[1]; te=m.evaluate(Xtest,ytest,verbose=0)[1]
    results.append([kind,param,tr,te,tr-te])
print(pd.DataFrame(results,columns=["Model","Setting","Train accuracy","Test accuracy","Train-test gap"]))